In [13]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

df.printSchema()
df.show(5)



root
 |-- _id: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+-------------------+----------------+--------------------+-----+
|                 _id|      fecha_captura|           grupo|       identificador|valor|
+--------------------+-------------------+----------------+--------------------+-----+
|69cd0dd3f269797d4...|2026-04-01 12:21:30|Nombre del grupo|A Light in the Attic|51.77|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|  Tipping the Velvet|53.74|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|          Soumission| 50.1|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|       Sharp Objects|47.82|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+----------------+--------------------+-----+
only showing

In [6]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("identificador").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 120
Registros validos: 120


In [8]:
df.select("identificador", "valor").show()

+--------------------+-----+
|       identificador|valor|
+--------------------+-----+
|A Light in the Attic|51.77|
|  Tipping the Velvet|53.74|
|          Soumission| 50.1|
|       Sharp Objects|47.82|
|Sapiens: A Brief ...|54.23|
|     The Requiem Red|22.65|
|The Dirty Little ...|33.34|
|The Coming Woman:...|17.93|
|The Boys in the B...| 22.6|
|     The Black Maria|52.15|
|Starving Hearts (...|13.99|
|Shakespeare's Son...|20.66|
|         Set Me Free|17.46|
|Scott Pilgrim's P...|52.29|
|Rip it Up and Sta...|35.02|
|Our Band Could Be...|57.25|
|                Olio|23.88|
|Mesaerion: The Be...|37.59|
|Libertarianism fo...|51.33|
|It's Only the Him...|45.17|
+--------------------+-----+
only showing top 20 rows



In [10]:
df.filter(df["valor"] >5).show()

+--------------------+-------------------+----------------+--------------------+-----+
|                 _id|      fecha_captura|           grupo|       identificador|valor|
+--------------------+-------------------+----------------+--------------------+-----+
|69cd0dd3f269797d4...|2026-04-01 12:21:30|Nombre del grupo|A Light in the Attic|51.77|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|  Tipping the Velvet|53.74|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|          Soumission| 50.1|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|       Sharp Objects|47.82|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|Sapiens: A Brief ...|54.23|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|     The Requiem Red|22.65|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|The Dirty Little ...|33.34|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|Nombre del grupo|The Coming Woman:...|17.93|
|69cd0dd3f269797d4...|2026-04-01 12:21:31|N

In [11]:
df.groupBy("grupo").count().show()

+----------------+-----+
|           grupo|count|
+----------------+-----+
|Nombre del grupo|  120|
+----------------+-----+



In [15]:
from pyspark.sql import functions as F
# 2. Transformacion y Agregacion por CATEGORIA
 # Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("grupo").agg(
F.count("identificador").alias("cantidad_libros"),
F.avg("valor").alias("precio_promedio"),
F.min("valor").alias("precio_minimo"),
F.max("valor").alias("precio_maximo")
 ).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+----------------+---------------+-----------------+-------------+-------------+
|           grupo|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+----------------+---------------+-----------------+-------------+-------------+
|Nombre del grupo|            120|35.00266666666667|        12.84|        57.31|
+----------------+---------------+-----------------+-------------+-------------+



In [16]:
from pyspark.sql import functions as F

# ordenar de mayor a menor
ganador = df.orderBy(F.desc("valor")).limit(1)

# mostrar info
ganador.select(
    "identificador",
    "valor",
    "grupo",
    "fecha_captura"
).show(truncate=False)

+------------------------------+-----+----------------+-------------------+
|identificador                 |valor|grupo           |fecha_captura      |
+------------------------------+-----+----------------+-------------------+
|Slow States of Collapse: Poems|57.31|Nombre del grupo|2026-04-01 12:21:36|
+------------------------------+-----+----------------+-------------------+



In [19]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "Nombre del grupo"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("grupo") \
    .agg(
        F.count("identificador").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: Nombre del grupo ---
+----------------+------------+------------+---------------------+
|           grupo|total_libros|precio_medio|ultima_sincronizacion|
+----------------+------------+------------+---------------------+
|Nombre del grupo|         120|        35.0|  2026-04-01 12:45:05|
+----------------+------------+------------+---------------------+

